## Configurando agente orquestrador A2A

No módulo anterior, iniciamos dois agentes, usando AgentCore Runtime, que suportam invocações A2A.

Neste laboratório, vamos adicionar um orquestrador, que invocará nossos sub-agentes.

<img src="images/architecture.png" style="width: 80%;">

Então vamos começar!

### Configuração

Importar dependências necessárias

In [ ]:
# Importar bibliotecas
import os
import json
import requests
import boto3
from boto3.session import Session
from strands.tools import tool

# Obter sessão boto
boto_session = Session()
region = boto_session.region_name

Recuperar informações dos LABs anteriores, para que possamos usá-las neste.

In [ ]:
%store -r

### 1 - Criar código para o agente orquestrador

Vamos gerar código Python que será usado para nosso orquestrador, e posteriormente será implantado no AgentCore.

In [ ]:
%%writefile agents/orchestrator.py
import logging
import json
import asyncio
from typing import Dict, Optional
from urllib.parse import quote
from uuid import uuid4

import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.types import Message, Part, Role, TextPart

from helpers.utils import get_cognito_secret, reauthenticate_user, get_ssm_parameter, SSM_DOCS_AGENT_ARN, SSM_BLOGS_AGENT_ARN

from strands import Agent, tool
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from fastapi import HTTPException

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Timeouts reduzidos para evitar travamento
DEFAULT_TIMEOUT = 15  # 15s em vez de 300s
AGENT_TIMEOUT = 10    # 10s por chamada de agente

# Cache global e pool de conexões
_cache = {
    'cognito_config': None,
    'agent_arns': {},
    'agent_cards': {},
    'http_client': None
}

app = BedrockAgentCoreApp()

def get_cached_config():
    """Cachear todas as operações caras"""
    if not _cache['agent_arns']:
        _cache['agent_arns'] = {
            'docs': get_ssm_parameter(SSM_DOCS_AGENT_ARN),
            'blogs': get_ssm_parameter(SSM_BLOGS_AGENT_ARN)
        }
    
    if not _cache['cognito_config']:
        secret = json.loads(get_cognito_secret())
        _cache['cognito_config'] = {
            'client_id': secret.get("client_id"),
            'client_secret': secret.get("client_secret")
        }
    
    return _cache['agent_arns'], _cache['cognito_config']

def get_bearer_token():
    """Gerar token bearer novo para cada requisição"""
    _, config = get_cached_config()
    return reauthenticate_user(
        config['client_id'], 
        config['client_secret']
    )

def get_http_client():
    """Reutilizar cliente HTTP com timeouts agressivos"""
    if not _cache['http_client']:
        _cache['http_client'] = httpx.AsyncClient(
            timeout=httpx.Timeout(DEFAULT_TIMEOUT, connect=5.0),
            limits=httpx.Limits(max_keepalive_connections=5, max_connections=10),
            http2=True  # Habilitar HTTP/2 para melhor desempenho
        )
    return _cache['http_client']

def create_message(text: str) -> Message:
    return Message(
        kind="message",
        role=Role.user,
        parts=[Part(TextPart(kind="text", text=text))],
        message_id=uuid4().hex,
    )

async def send_agent_message(message: str, agent_type: str) -> Optional[str]:
    """Comunicação otimizada com agente com padrão circuit breaker"""
    try:
        agent_arns, _ = get_cached_config()
        agent_arn = agent_arns[agent_type]
        bearer_token = get_bearer_token()
        
        from boto3.session import Session
        region = Session().region_name
        
        escaped_arn = quote(agent_arn, safe='')
        runtime_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations/"
        
        headers = {
            "Authorization": f"Bearer {bearer_token}",
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': str(uuid4())
        }
        
        httpx_client = get_http_client()
        httpx_client.headers.update(headers)
        
        # Cachear agent card
        if agent_arn not in _cache['agent_cards']:
            resolver = A2ACardResolver(httpx_client=httpx_client, base_url=runtime_url)
            _cache['agent_cards'][agent_arn] = await asyncio.wait_for(
                resolver.get_agent_card(), timeout=5.0
            )
        
        agent_card = _cache['agent_cards'][agent_arn]
        
        # Criar cliente com modo não-streaming
        config = ClientConfig(httpx_client=httpx_client, streaming=False)
        factory = ClientFactory(config)
        client = factory.create(agent_card)
        
        msg = create_message(message)
        
        # Usar timeout para toda a operação
        async with asyncio.timeout(AGENT_TIMEOUT):
            async for event in client.send_message(msg):
                if isinstance(event, Message):
                    return event.parts[0].text if event.parts else "Sem resposta"
                elif isinstance(event, tuple) and len(event) == 2:
                    return event[0].parts[0].text if event[0].parts else "Sem resposta"
        
        return "Timeout: Nenhuma resposta recebida"
        
    except asyncio.TimeoutError:
        logger.warning(f"Timeout ao chamar agente {agent_type}")
        return f"Agente {agent_type} expirou"
    except Exception as e:
        logger.error(f"Erro ao chamar {agent_type}: {e}")
        return f"Erro: {str(e)[:100]}"

@tool
async def send_mcp_message(message: str):
    """Enviar mensagem para agente AWS Docs com timeout"""
    return await send_agent_message(f"Resuma brevemente: {message}", 'docs')

@tool
async def send_blog_message(message: str):
    """Enviar mensagem para agente AWS Blogs com timeout"""
    return await send_agent_message(f"Resuma brevemente: {message}", 'blogs')


system_prompt = """Você é um orquestrador de informações AWS.

Agentes disponíveis:
- Documentação AWS: Detalhes técnicos de serviços AWS
- Blogs AWS: Últimas notícias e anúncios AWS

IMPORTANTE: Mantenha respostas CURTAS e RÁPIDAS. Sempre solicite resumos dos sub-agentes.

Diretrizes:
- Use consultas paralelas quando possível
- Timeout após 10 segundos por agente
- Forneça respostas rápidas e práticas
- Se agentes expirarem, forneça o que você sabe
"""

agent = Agent(
    system_prompt=system_prompt, 
    tools=[send_mcp_message, send_blog_message],
    name="AWS Orchestration Agent",
    description="Um agente para orquestrar sub-agentes"
)

@app.entrypoint
async def invoke_agent(payload, context):
    logger.info("Orquestrador rápido processando requisição")
    
    try:
        user_prompt = payload.get("prompt", "")
        if not user_prompt:
            raise HTTPException(status_code=400, detail="Nenhum prompt fornecido")

        logger.info(f"Consulta: {user_prompt[:100]}...")
        
        # Definir timeout geral para toda a operação
        async with asyncio.timeout(25.0):  # Máximo 25s no total
            agent_stream = agent.stream_async(user_prompt)
            
            async for event in agent_stream:
                yield event

    except asyncio.TimeoutError:
        logger.error("Operação geral expirou")
        yield {"error": "Requisição expirou após 25 segundos"}
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f"Processamento falhou: {e}")
        yield {"error": f"Processamento falhou: {str(e)[:100]}"}

# Limpeza no desligamento
async def cleanup():
    if _cache['http_client']:
        await _cache['http_client'].aclose()

if __name__ == "__main__":
    import atexit
    atexit.register(lambda: asyncio.run(cleanup()))
    app.run()

#### 1.1 - Criar função IAM para o agente

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, ORCHESTRATOR_ROLE_NAME

agent_name="aws_orchestrator_assistant"

execution_role_arn = create_agentcore_runtime_execution_role(ORCHESTRATOR_ROLE_NAME)

### 2 - Implantar no AgentCore Runtime

Agora, vamos implantar o orquestrador no AgentCore Runtime.

Note que neste exemplo, não estamos adicionando o parâmetro `protocol`. Isso significa que este será um agente HTTP.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

# Configurar a implantação
response = agentcore_runtime.configure(
    entrypoint="agents/orchestrator.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [COGNITO_CLIENT_ID],
            "discoveryUrl": DISCOVERY_URL,
        }
    },
)

print("Configuração concluída:", response)

In [ ]:
launch_result = agentcore_runtime.launch()
print("Inicialização concluída:", launch_result.agent_arn)

agent_arn = launch_result.agent_arn

**Verificar status da implantação**

Vamos verificar se a implantação foi concluída:

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]

print(f"Status final: {status}")

#### 2.1 - Exportar e salvar saídas

Exportar variáveis para serem usadas no notebook de limpeza:

In [ ]:
ORCHESTRATION_ID = launch_result.agent_id
ORCHESTRATION_ARN = launch_result.agent_arn
ORCHESTRATION_NAME = agent_name

%store ORCHESTRATION_ID
%store ORCHESTRATION_ARN
%store ORCHESTRATION_NAME

### 3 - Invocar agentes A2A usando um agente orquestrador

Primeiro, vamos atualizar o token de autenticação:

In [ ]:
from helpers.utils import reauthenticate_user

bearer_token = reauthenticate_user(
    COGNITO_CLIENT_ID,
    COGNITO_SECRET
)

Agora, vamos invocar nosso orquestrador para verificar AWS Docs, fazendo uma chamada ao nosso primeiro agente, usando A2A:

In [ ]:
import requests
import json
import uuid
from urllib.parse import quote

session_id = str(uuid.uuid4())
print(f'Invocando para sessão: {session_id}')

headers = {
    'Authorization': f'Bearer {bearer_token}',
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id
}

prompt = {"prompt": "O que é DynamoDB?"}

escaped_agent_arn = quote(ORCHESTRATION_ARN, safe='')

response = requests.post(
    f'https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations',
    headers=headers,
    data=json.dumps(prompt)
)

for line in response.iter_lines(decode_unicode=True):
    if line.startswith('data: '):
        data = line[6:]
        try:
            parsed = json.loads(data)
            print(parsed)
        except:
            print(data)

In [ ]:
import uuid

session_id = str(uuid.uuid4())
print(f'Invocando para sessão: {session_id}')

headers = {
    'Authorization': f'Bearer {bearer_token}',
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id
}

prompt = {"prompt": "Me dê o último blog publicado sobre Bedrock AgentCore?"}

escaped_agent_arn = quote(ORCHESTRATION_ARN, safe='')

response = requests.post(
    f'https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations',
    headers=headers,
    data=json.dumps(prompt)
)

for line in response.iter_lines(decode_unicode=True):
    if line.startswith('data: '):
        data = line[6:]
        try:
            parsed = json.loads(data)
            print(parsed)
        except:
            print(data)

Parabéns, você implantou a solução completa, usando protocolo A2A no Amazon AgentCore Runtime.